In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
midfielders = pd.read_csv(
    "data\processed\midfielders_clustered.csv"
)

forwards = pd.read_csv(
    "data\processed\ forwards_clustered.csv"
)

defenders = pd.read_csv(
    "data\processed\defenders_clustered.csv"
)




<>:2: SyntaxWarning: invalid escape sequence '\p'
<>:6: SyntaxWarning: invalid escape sequence '\p'
<>:10: SyntaxWarning: invalid escape sequence '\p'
<>:2: SyntaxWarning: invalid escape sequence '\p'
<>:6: SyntaxWarning: invalid escape sequence '\p'
<>:10: SyntaxWarning: invalid escape sequence '\p'
C:\Users\Titus\AppData\Local\Temp\ipykernel_31152\1452853412.py:2: SyntaxWarning: invalid escape sequence '\p'
  "data\processed\midfielders_clustered.csv"
C:\Users\Titus\AppData\Local\Temp\ipykernel_31152\1452853412.py:6: SyntaxWarning: invalid escape sequence '\p'
  "data\processed\ forwards_clustered.csv"
C:\Users\Titus\AppData\Local\Temp\ipykernel_31152\1452853412.py:10: SyntaxWarning: invalid escape sequence '\p'
  "data\processed\defenders_clustered.csv"
C:\Users\Titus\AppData\Local\Temp\ipykernel_31152\1452853412.py:1: DtypeWarning: Columns (0: weak_foot) have mixed types. Specify dtype option on import or set low_memory=False.
  midfielders = pd.read_csv(


In [6]:

midfielder_features = [
    #Scoring
    "goals_per90",
    "xg_per90",

    #Chance Creation
    "assists_per90",
    "xag_per90",
    "per_90_minutes_kp",
    "sca_sca90",

    #Progression
    "per_90_minutes_prgp",
    "per_90_minutes_prgc",
    "per_90_minutes_1_per_3",
    "per_90_minutes_ppa",


    #Possession/passing
    "pass_completion_pct",

    #Defense
    "per_90_minutes_tackles_tkl",
    "per_90_minutes_int"

]

forward_features = [
    # Scoring
    "goals_per90",
    "xg_per90",

    # Chance Creation
    "assists_per90",
    "xag_per90",
    "per_90_minutes_kp",
    "sca_sca90",

    # Progression / Attacking
    "per_90_minutes_prgc",
    "per_90_minutes_ppa",

    # Possession
    "pass_completion_pct",

    # Defensive Work
    "per_90_minutes_tackles_tkl",
    "per_90_minutes_int"
]

defender_features = [
    # Defensive Contribution
    "per_90_minutes_tackles_tkl",
    "per_90_minutes_int",

    # Passing
    "pass_completion_pct",

    # Progression
    "per_90_minutes_prgp",
    "per_90_minutes_prgc",
    "per_90_minutes_1_per_3",

    # Attacking Contribution
    "assists_per90",
    "xag_per90",
    "per_90_minutes_kp",

    # Optional scoring contribution
    "goals_per90",
    "xg_per90"
]

POSITION_FEATURES = {
    "Midfielder": midfielder_features,
    "Forward": forward_features,
    "Defender": defender_features
}

position_data = {
    "Midfielder": midfielders,
    "Forward": forwards,
    "Defender": defenders
}



In [7]:
similarity_sets = {}
similarity_imputers = {}
similarity_scalers = {}

for position, dataframe in position_data.items():

    features = POSITION_FEATURES[position]

    X = dataframe[
        features
    ].copy()

    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    )

    imputer = SimpleImputer(
        strategy="median"
    )

    X_imputed = imputer.fit_transform(
        X
    )

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(
        X_imputed
    )

    similarity_sets[position] = X_scaled
    similarity_imputers[position] = imputer
    similarity_scalers[position] = scaler

    print(
        position,
        X_scaled.shape
    )

Midfielder (7874, 13)
Forward (5723, 11)
Defender (9525, 11)


In [8]:
#Similarity Function
def find_similar_players(
    player_name,
    position,
    position_data,
    similarity_sets,
    n=5,
    same_archetype=True,
    season=None
):

    dataframe = position_data[position]

    X_scaled = similarity_sets[position]

    player_rows = dataframe[
        dataframe["player"] == player_name
    ]

    if season is not None:
        player_rows = player_rows[
            player_rows["season"] == season
        ]

    if player_rows.empty:
        raise ValueError(
            f"{player_name} not found"
        )

    player_index = player_rows.index[-1]

    query_position = (
        dataframe.index.get_loc(
            player_index
        )
    )

    similarities = cosine_similarity(
        X_scaled[
            query_position
        ].reshape(1, -1),
        X_scaled
    )[0]

    results = dataframe.copy()

    results["similarity"] = similarities

    results = results[
        results["player"] != player_name
    ]

    if same_archetype:

        archetype = dataframe.loc[
            player_index,
            "archetype"
        ]

        results = results[
            results["archetype"] == archetype
        ]

    results = (
        results
        .sort_values(
            "similarity",
            ascending=False
        )
        .drop_duplicates(
            subset="player"
        )
        .head(n)
    )

    return results[
        [
            "player",
            "team",
            "season",
            "player_positions",
            "archetype",
            "similarity"
        ]
    ]






In [14]:
#test creative midfielder similar players
find_similar_players(
    "Rodri",
    "Midfielder",
    position_data,
    similarity_sets,
    n=5,
    season=2324
)

,player,team,season,player_positions,archetype,similarity
553,Andre Horta,Braga,2324,"CM, CDM",Creative / Progressive Midfielder,0.948515
7359,Toni Kroos,Real Madrid,1920,CM,Creative / Progressive Midfielder,0.935989
4595,Luka Modric,Real Madrid,1819,"CM, CDM",Creative / Progressive Midfielder,0.931258
1391,Corentin Tolisso,Bayern Munich,1718,"CM, CDM",Creative / Progressive Midfielder,0.925534
2019,Eduard Atuesta,LAFC,1920,"CDM, CM",Creative / Progressive Midfielder,0.925473


In [15]:
#Test defensive midfielder similar players
find_similar_players(
    "Erling Haaland",
    "Forward",
    position_data,
    similarity_sets,
    n=5,
    season=2324
)


,player,team,season,player_positions,archetype,similarity
4453,Pierre-Emerick Aubameyang,Dortmund,1718,ST,Direct / Finishing Forward,0.985950
2731,Josef Martinez,Atlanta Utd,1819,ST,Direct / Finishing Forward,0.982057
355,Andre Silva,Eint Frankfurt,2021,ST,Direct / Finishing Forward,0.974810
4123,Nikola Kalinic,Roma,1920,ST,Direct / Finishing Forward,0.973227
5016,Serhou Guirassy,Dortmund,2425,ST,Direct / Finishing Forward,0.971621


In [22]:
#Test midfielder similar players without archetype restriction
find_similar_players(
    "Cristiano Ronaldo",
    "Forward",
    position_data,
    similarity_sets,
    n=5,
    season=2223
)

,player,team,season,player_positions,archetype,similarity
4560,Rasmus Højlund,Manchester Utd,2425,ST,Direct / Finishing Forward,0.965483
2217,Ibrahima Niane,Angers,2223,"ST, LW",Direct / Finishing Forward,0.949584
1835,Francesco Caputo,Empoli,2324,ST,Direct / Finishing Forward,0.946756
4452,Pierre-Emerick Aubameyang,Chelsea,2223,ST,Direct / Finishing Forward,0.939085
4915,Samuele Mulattieri,Sassuolo,2324,ST,Direct / Finishing Forward,0.932082


In [110]:
#Turn similarity score into percentage
results = find_similar_players(
    "Kevin De Bruyne",
    midfielders,
    X_scaled,
    n=5
)

results["similarity_pct"] = (
    results["similarity"] * 100
).round(1)

print(results)


                   player                team  season player_positions  \
7070      Steven Berghuis                Ajax    2122               RW   
3912         Josip Ilicic            Atalanta    2021           CF, ST   
983          Borek Dockal  Philadelphia Union    1819          CAM, CM   
904   Benjamin Bourigeaud              Rennes    2223       RW, RM, LM   
1166           Carles Gil       NE Revolution    2021      RM, CAM, CM   

                              archetype  similarity  similarity_pct  
7070  Creative / Progressive Midfielder    0.978456            97.8  
3912  Creative / Progressive Midfielder    0.976570            97.7  
983   Creative / Progressive Midfielder    0.970568            97.1  
904   Creative / Progressive Midfielder    0.963427            96.3  
1166  Creative / Progressive Midfielder    0.963045            96.3  


In [24]:
def compare_players(
    player1,
    player2,
    position,
    position_data,
    position_features
):

    dataframe = position_data[position]
    features = position_features[position]

    p1_rows = dataframe[
        dataframe["player"].str.lower()
        == player1.lower()
    ]

    p2_rows = dataframe[
        dataframe["player"].str.lower()
        == player2.lower()
    ]

    if p1_rows.empty:
        raise ValueError(
            f"{player1} not found in {position} data."
        )

    if p2_rows.empty:
        raise ValueError(
            f"{player2} not found in {position} data."
        )

    p1 = (
        p1_rows
        .sort_values(
            "season",
            ascending=False
        )
        .iloc[0]
    )

    p2 = (
        p2_rows
        .sort_values(
            "season",
            ascending=False
        )
        .iloc[0]
    )

    comparison = pd.DataFrame({
        "Feature": features,

        player1: [
            p1[f]
            for f in features
        ],

        player2: [
            p2[f]
            for f in features
        ]
    })

    return comparison

comparison = compare_players(
    "Kevin De Bruyne",
    "Bruno Fernandes",
    "Midfielder",
    position_data,
    POSITION_FEATURES
)

print(comparison)

comparison = compare_players(
    "Cristiano Ronaldo",
    "Robert Lewandowski",
    "Forward",
    position_data,
    POSITION_FEATURES
)

print(comparison)

comparison = compare_players(
    "Virgil van Dijk",
    "Ruben Dias",
    "Defender",
    position_data,
    POSITION_FEATURES
)

print(comparison)



                       Feature  Kevin De Bruyne  Bruno Fernandes
0                  goals_per90         0.210000         0.240000
1                     xg_per90         0.270000         0.300000
2                assists_per90         0.370000         0.300000
3                    xag_per90         0.420000         0.250000
4            per_90_minutes_kp         3.015873         2.716418
5                    sca_sca90         5.970000         5.760000
6          per_90_minutes_prgp         8.306878         9.701493
7          per_90_minutes_prgc         2.698413         2.298507
8       per_90_minutes_1_per_3         4.656085         6.388060
9           per_90_minutes_ppa         3.121693         2.537313
10         pass_completion_pct        75.900000        76.100000
11  per_90_minutes_tackles_tkl         0.899471         2.507463
12          per_90_minutes_int         0.370370         0.776119
                       Feature  Cristiano Ronaldo  Robert Lewandowski
0                  g

In [25]:
for position, dataframe in position_data.items():

    features = POSITION_FEATURES[position]

    for feature in features:

        percentile_name = (
            feature + "_percentile"
        )

        dataframe[percentile_name] = (
            dataframe
            .groupby("season")[feature]
            .rank(
                pct=True
            )
            .mul(100)
        )

    print(
        f"{position} percentiles complete."
    )

Midfielder percentiles complete.
Forward percentiles complete.
Defender percentiles complete.


C:\Users\Titus\AppData\Local\Temp\ipykernel_31152\2483722828.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataframe[percentile_name] = (
C:\Users\Titus\AppData\Local\Temp\ipykernel_31152\2483722828.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataframe[percentile_name] = (
C:\Users\Titus\AppData\Local\Temp\ipykernel_31152\2483722828.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns 

In [26]:
for position, dataframe in position_data.items():

    percentile_cols = [
        feature + "_percentile"
        for feature
        in POSITION_FEATURES[position]
    ]

    print(
        f"\n===== {position} ====="
    )

    print(
        dataframe[
            [
                "player",
                "season",
                "archetype"
            ]
            + percentile_cols
        ].head()
    )


===== Midfielder =====
            player  season                          archetype  \
0  Aaron Boupendza    2021     Defensive / Holding Midfielder   
1      Aaron Kovar    1819  Creative / Progressive Midfielder   
2     Aaron Lennon    1718     Defensive / Holding Midfielder   
3     Aaron Lennon    1819     Defensive / Holding Midfielder   
4     Aaron Lennon    1920     Defensive / Holding Midfielder   

   goals_per90_percentile  xg_per90_percentile  assists_per90_percentile  \
0              100.000000                  NaN                 33.713200   
1               13.561077            81.807229                 95.445135   
2               15.488810            28.998073                 73.085984   
3               41.459627            32.469880                 42.391304   
4               15.539859             3.150525                 14.026236   

   xag_per90_percentile  per_90_minutes_kp_percentile  sca_sca90_percentile  \
0                   NaN                          

In [27]:
#midfielders.to_csv("midfielders_similarity_ready.csv", index=False)

for position, dataframe in position_data.items():

    filename = (
        f"data/processed/ "
        f"{position.lower()}s_similarity_ready.csv"
    )

    dataframe.to_csv(
        filename,
        index=False
    )

    print(
        f"Saved {position}: {filename}"
    )


Saved Midfielder: data/processed/ midfielders_similarity_ready.csv
Saved Forward: data/processed/ forwards_similarity_ready.csv
Saved Defender: data/processed/ defenders_similarity_ready.csv
